#00. Imports & Config

In [0]:
from __future__ import annotations

from typing import Dict, List, Optional, Tuple
import pandas as pd
import numpy as np
import re
import json

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [0]:
def get_str(name: str) -> str:
    return dbutils.widgets.get(name).strip()

def get_int(name: str) -> int:
    return int(get_str(name))

def get_float(name: str) -> float:
    return float(get_str(name))

def get_bool(name: str) -> bool:
    v = get_str(name).lower()
    if v in ("true", "1", "yes", "y", "t"):  return True
    if v in ("false", "0", "no", "n", "f"): return False
    raise ValueError(f"Invalid boolean for {name}: {v}")

def get_json(name: str):
    return json.loads(get_str(name))

In [0]:
dbutils.widgets.text("ORIGIN_TABLE", "samples.bakehouse.sales_transactions")
dbutils.widgets.text("DATE_COL", "dateTime")
dbutils.widgets.text("QTY_COL", "quantity")

dbutils.widgets.text("KEY_COLS", json.dumps(["product"]))
dbutils.widgets.text("KEY_DELIM", ";")

dbutils.widgets.text("MACRO_FORECASTING_TABLE", "workspace.forecasting.macro_forecast")
dbutils.widgets.text("MACRO_KEY_COL", "cd_key")

dbutils.widgets.text("ALIAS_DT", "dt_period")
dbutils.widgets.text("ALIAS_KEY", "cd_key")
dbutils.widgets.text("ALIAS_UNITS", "qt_units")

dbutils.widgets.text("TOP_N", "10")

dbutils.widgets.text("PANDEMIC_START", "2020-03-01")
dbutils.widgets.text("PANDEMIC_END", "2021-12-01")
dbutils.widgets.text("POST_PANDEMIC_START", "2022-01-01")

dbutils.widgets.text("TYPE_PERIOD", "day")
dbutils.widgets.text("DOW_BOOLEAN", "false")

dbutils.widgets.text("RANGE_PREDICTION", "5")
dbutils.widgets.text("VALIDATION_RANGE", "2")

dbutils.widgets.text("PERIOD_COLUMN", "dt_period")
dbutils.widgets.text("ID_COLUMN", "cd_key")
dbutils.widgets.text("Y_COLUMN", "qt_units")

dbutils.widgets.text("CV_FOLDS", "1")
dbutils.widgets.text("MIN_TRAIN_PERIOD", "3")

dbutils.widgets.text("FILL_MISSING_VALUES", "0.0")
dbutils.widgets.text("CLIP_NEGATIVE", "true")

In [0]:
ORIGIN_TABLE = get_str("ORIGIN_TABLE")
DATE_COL = get_str("DATE_COL")
QTY_COL = get_str("QTY_COL")

KEY_COLS = get_json("KEY_COLS")
KEY_DELIM = get_str("KEY_DELIM")

MACRO_FORECASTING_TABLE = get_str("MACRO_FORECASTING_TABLE")
MACRO_KEY_COL = get_str("MACRO_KEY_COL")

ALIAS_DT = get_str("ALIAS_DT")
ALIAS_KEY = get_str("ALIAS_KEY")
ALIAS_UNITS = get_str("ALIAS_UNITS")

TOP_N = get_int("TOP_N")

PANDEMIC_START = get_str("PANDEMIC_START")
PANDEMIC_END = get_str("PANDEMIC_END")
POST_PANDEMIC_START = get_str("POST_PANDEMIC_START")

TYPE_PERIOD = get_str("TYPE_PERIOD")
DOW_BOOLEAN = get_bool("DOW_BOOLEAN")

RANGE_PREDICTION = get_int("RANGE_PREDICTION")
VALIDATION_RANGE = get_int("VALIDATION_RANGE")

PERIOD_COLUMN = get_str("PERIOD_COLUMN")
ID_COLUMN = get_str("ID_COLUMN")
Y_COLUMN = get_str("Y_COLUMN")

CV_FOLDS = get_int("CV_FOLDS")
MIN_TRAIN_PERIOD = get_int("MIN_TRAIN_PERIOD")

FILL_MISSING_VALUES = get_float("FILL_MISSING_VALUES")
CLIP_NEGATIVE = get_bool("CLIP_NEGATIVE")

In [0]:
current_date = spark.sql(f"select max(TRY_CAST(dateTime AS DATE)) from {ORIGIN_TABLE}").collect()[0][0]

In [0]:
spark.sql(f"""
  CREATE TABLE IF NOT EXISTS {MACRO_FORECASTING_TABLE} (
  cd_key STRING,
  record_type STRING, 
  split STRING,
  model_id STRING,
  effective_fh INTEGER,
  dt_period DATE,
  y_true DOUBLE,
  y_pred DOUBLE,
  mape DOUBLE,
  winner_model_id STRING,
  winner_mape_recent DOUBLE,
  status STRING
)
""")

#01. Input

In [0]:
def safe_table_ident(name: str) -> str:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*(\.[A-Za-z_][A-Za-z0-9_]*){0,2}", name):
        raise ValueError(f"Invalid table identifier: {name}")
    return name

def safe_ident(name: str) -> str:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", name):
        raise ValueError(f"Invalid identifier: {name}")
    return name

def safe_sql_string_literal(s: str) -> str:
    if not isinstance(s, str) or len(s) > 10:
        raise ValueError("Invalid delimiter")
    if re.search(r"[\x00-\x1f\x7f]", s):
        raise ValueError("Invalid delimiter")
    return "'" + s.replace("'", "''") + "'"

def build_key_expr(key_cols: list[str], delimiter: str = ";") -> tuple[str, str]:
    if not key_cols:
        raise ValueError("key_cols cannot be empty")

    cols = [safe_ident(c) for c in key_cols]
    delim_lit = safe_sql_string_literal(delimiter)

    cols_str = [f"CAST({c} AS STRING)" for c in cols]

    if len(cols_str) == 1:
        expr = cols_str[0]
    else:
        expr = f"concat_ws({delim_lit}, {', '.join(cols_str)})"

    return expr, expr

def safe_int(n) -> int:
    n = int(n)
    if n <= 0 or n > 10_000:
        raise ValueError("Invalid integer range")
    return n

In [0]:
def create_input():
    t_origin = safe_table_ident(ORIGIN_TABLE)
    t_macro  = safe_table_ident(MACRO_FORECASTING_TABLE)

    c_date = safe_ident(DATE_COL)
    c_qty  = safe_ident(QTY_COL)
    c_macro_key = safe_ident(MACRO_KEY_COL)

    a_dt    = safe_ident(ALIAS_DT)
    a_key   = safe_ident(ALIAS_KEY)
    a_units = safe_ident(ALIAS_UNITS)

    top_n = safe_int(TOP_N)

    key_expr, key_groupby = build_key_expr(KEY_COLS, KEY_DELIM)

    query = f"""
    WITH macro_keys AS (
    SELECT DISTINCT CAST({c_macro_key} AS STRING) AS macro_cd_key
    FROM {t_macro}
    WHERE {c_macro_key} IS NOT NULL
    ),
    fato AS (
    SELECT
    TRY_CAST({c_date} AS DATE) AS {a_dt},
    {key_expr} AS {a_key},
    SUM({c_qty}) AS {a_units}
    FROM {t_origin}
    GROUP BY TRY_CAST({c_date} AS DATE), {key_groupby}
    ),
    lista AS (
    SELECT
    rank() OVER (ORDER BY SUM({c_qty}) DESC) AS rank,
    {key_expr} AS {a_key}
    FROM {t_origin} o
    LEFT ANTI JOIN macro_keys m ON {key_expr} = m.macro_cd_key
    GROUP BY {key_groupby}
    ORDER BY rank ASC
    LIMIT {top_n}
    )
    SELECT f.*
    FROM fato f
    WHERE f.{a_key} IN (SELECT l.{a_key} FROM lista l)
    """
    df_input = spark.sql(query)

    return df_input

#02. Metrics

In [0]:
def period_cfg(type_period: str) -> dict:
    tp = (type_period or "").strip().lower()

    base = {
        "day": {
            "spark_trunc": "DAY",
            "pandas_freq": "D",
            "seasonal_periods": 7,
        },
        "daily": {
            "spark_trunc": "DAY",
            "pandas_freq": "D",
            "seasonal_periods": 7,
        },
        "week": {
            "spark_trunc": "WEEK",
            "pandas_freq": "W-MON",
            "seasonal_periods": 52, 
        },
        "weekly": {
            "spark_trunc": "WEEK",
            "pandas_freq": "W-MON",
            "seasonal_periods": 52,
        },
        "month": {
            "spark_trunc": "MONTH",
            "pandas_freq": "MS",
            "seasonal_periods": 12,
        },
        "monthly": {
            "spark_trunc": "MONTH",
            "pandas_freq": "MS",
            "seasonal_periods": 12,
        },
        "quarter": {
            "spark_trunc": "QUARTER",
            "pandas_freq": "QS",
            "seasonal_periods": 4,
        },
        "quarterly": {
            "spark_trunc": "QUARTER",
            "pandas_freq": "QS",
            "seasonal_periods": 4,
        },
        "year": {
            "spark_trunc": "YEAR",
            "pandas_freq": "YS",
            "seasonal_periods": 1,
        },
        "yearly": {
            "spark_trunc": "YEAR",
            "pandas_freq": "YS",
            "seasonal_periods": 1,
        },
        "annual": {
            "spark_trunc": "YEAR",
            "pandas_freq": "YS",
            "seasonal_periods": 1,
        },
    }

    if tp not in base:
        raise ValueError(f"Unsupported TYPE_PERIOD={type_period!r}. Use one of {list(base)}")

    cfg = dict(base[tp])

    spark_trunc = cfg["spark_trunc"].upper()
    freq = cfg["pandas_freq"]
    cycle = int(cfg.get("seasonal_periods", 1) or 1)

    # calendar driver for cyclical encoding
    if spark_trunc == "DAY":
        cycle_col = "dow"            # 0..6
        add_dow = DOW_BOOLEAN              
    elif spark_trunc == "WEEK":
        cycle_col = "weekofyear"     # 1..52/53
        add_dow = DOW_BOOLEAN
    elif spark_trunc == "MONTH":
        cycle_col = "month"          # 1..12
        add_dow = DOW_BOOLEAN
    elif spark_trunc == "QUARTER":
        cycle_col = "quarter"        # 1..4
        add_dow = DOW_BOOLEAN
    else:  # YEAR
        cycle_col = None
        add_dow = DOW_BOOLEAN

    cfg["cycle"] = cycle
    cfg["cycle_col"] = cycle_col
    cfg["add_dow"] = add_dow
    cfg["offset"] = pd.tseries.frequencies.to_offset(freq)

    # OPTIONAL defaults (tweak as you wish)
    if spark_trunc == "DAY":
        cfg["max_lag_default"] = 28
        cfg["rolling_windows_default"] = (7, 14, 28)
    elif spark_trunc == "WEEK":
        cfg["max_lag_default"] = 26
        cfg["rolling_windows_default"] = (4, 8, 13)
    elif spark_trunc == "MONTH":
        cfg["max_lag_default"] = 12
        cfg["rolling_windows_default"] = (3, 6, 12)
    elif spark_trunc == "QUARTER":
        cfg["max_lag_default"] = 8
        cfg["rolling_windows_default"] = (2, 4, 8)
    else:  # YEAR
        cfg["max_lag_default"] = 5
        cfg["rolling_windows_default"] = (2, 3, 5)

    return cfg

cfg = period_cfg(TYPE_PERIOD)

FREQ = cfg["pandas_freq"]
SPARK_TRUNC = cfg["spark_trunc"]
SEASONAL_PERIODS_WINDOW = cfg["seasonal_periods"]

CYCLE = cfg["cycle"]
CYCLE_COL = cfg["cycle_col"]
ADD_DOW = cfg["add_dow"]
OFFSET = cfg["offset"]

MAX_LAG_DEFAULT = cfg["max_lag_default"]
ROLLING_WINDOWS_DEFAULT = cfg["rolling_windows_default"]

In [0]:
def _safe_mape(y_true: np.ndarray, y_pred: np.ndarray, eps_abs: float = 1.0) -> float:
    y_true = np.asarray(y_true, dtype="float64")
    y_pred = np.asarray(y_pred, dtype="float64")
    denom = np.maximum(np.abs(y_true), eps_abs)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)


#03. Feature engineering for ML models (per-series)

In [0]:
def _build_supervised_matrix(
    y: pd.Series,
    max_lag: int | None = None,
    rolling_windows: Tuple[int, ...] | None = None,
) -> Tuple[pd.DataFrame, pd.Series, List[str]]:

    if max_lag is None:
        max_lag = int(MAX_LAG_DEFAULT)
    if rolling_windows is None:
        rolling_windows = tuple(ROLLING_WINDOWS_DEFAULT)

    df = y.to_frame("y").copy()
    df.index = pd.to_datetime(df.index)
    df["t"] = np.arange(len(df), dtype=np.int64)

    # --- calendar features ---
    trunc = str(SPARK_TRUNC).upper()

    if trunc == "DAY":
        dow = df.index.dayofweek.astype(int)  # 0=Mon..6=Sun
        if ADD_DOW:
            df["dow"] = dow

        # full 7-day cycle encoding
        if CYCLE and int(CYCLE) > 1:
            df["sin_cycle"] = np.sin(2.0 * np.pi * dow / float(CYCLE))
            df["cos_cycle"] = np.cos(2.0 * np.pi * dow / float(CYCLE))

    elif trunc == "WEEK":
        wk = df.index.isocalendar().week.astype(int)
        # optional raw feature
        df["weekofyear"] = wk
        if CYCLE and int(CYCLE) > 1:
            df["sin_cycle"] = np.sin(2.0 * np.pi * wk / float(CYCLE))
            df["cos_cycle"] = np.cos(2.0 * np.pi * wk / float(CYCLE))

    elif trunc == "MONTH":
        m = df.index.month.astype(int)
        df["month"] = m
        if CYCLE and int(CYCLE) > 1:
            df["sin_cycle"] = np.sin(2.0 * np.pi * m / float(CYCLE))
            df["cos_cycle"] = np.cos(2.0 * np.pi * m / float(CYCLE))

    elif trunc == "QUARTER":
        q = df.index.quarter.astype(int)
        df["quarter"] = q
        if CYCLE and int(CYCLE) > 1:
            df["sin_cycle"] = np.sin(2.0 * np.pi * q / float(CYCLE))
            df["cos_cycle"] = np.cos(2.0 * np.pi * q / float(CYCLE))

    # --- lags ---
    for lag in range(1, max_lag + 1):
        df[f"lag_{lag}"] = df["y"].shift(lag)

    # --- rolling stats (past-only) ---
    for w in rolling_windows:
        df[f"roll_mean_{w}"] = df["y"].shift(1).rolling(w).mean()
        df[f"roll_std_{w}"] = df["y"].shift(1).rolling(w).std()

    df = df.dropna()
    feature_cols = [c for c in df.columns if c != "y"]
    return df[feature_cols], df["y"], feature_cols


def _make_next_features(
    history: List[float],
    t_next: int,
    dt_next: pd.Timestamp,
    max_lag: int | None = None,
    rolling_windows: Tuple[int, ...] | None = None,
) -> Dict[str, float]:

    if max_lag is None:
        max_lag = int(MAX_LAG_DEFAULT)
    if rolling_windows is None:
        rolling_windows = tuple(ROLLING_WINDOWS_DEFAULT)

    feats: Dict[str, float] = {"t": float(t_next)}
    ts = pd.Timestamp(dt_next)

    trunc = str(SPARK_TRUNC).upper()

    # --- calendar features ---
    if trunc == "DAY":
        dow = int(ts.dayofweek)
        if ADD_DOW:
            feats["dow"] = float(dow)
        if CYCLE and int(CYCLE) > 1:
            feats["sin_cycle"] = float(np.sin(2.0 * np.pi * dow / float(CYCLE)))
            feats["cos_cycle"] = float(np.cos(2.0 * np.pi * dow / float(CYCLE)))

    elif trunc == "WEEK":
        wk = int(ts.isocalendar().week)
        feats["weekofyear"] = float(wk)
        if CYCLE and int(CYCLE) > 1:
            feats["sin_cycle"] = float(np.sin(2.0 * np.pi * wk / float(CYCLE)))
            feats["cos_cycle"] = float(np.cos(2.0 * np.pi * wk / float(CYCLE)))

    elif trunc == "MONTH":
        m = int(ts.month)
        feats["month"] = float(m)
        if CYCLE and int(CYCLE) > 1:
            feats["sin_cycle"] = float(np.sin(2.0 * np.pi * m / float(CYCLE)))
            feats["cos_cycle"] = float(np.cos(2.0 * np.pi * m / float(CYCLE)))

    elif trunc == "QUARTER":
        q = int(ts.quarter)
        feats["quarter"] = float(q)
        if CYCLE and int(CYCLE) > 1:
            feats["sin_cycle"] = float(np.sin(2.0 * np.pi * q / float(CYCLE)))
            feats["cos_cycle"] = float(np.cos(2.0 * np.pi * q / float(CYCLE)))

    # --- lags ---
    for lag in range(1, max_lag + 1):
        feats[f"lag_{lag}"] = float(history[-lag]) if len(history) >= lag else float("nan")

    # --- rolling windows ---
    for w in rolling_windows:
        if len(history) >= w:
            window_vals = np.asarray(history[-w:], dtype="float64")
            feats[f"roll_mean_{w}"] = float(np.mean(window_vals))
            feats[f"roll_std_{w}"] = float(np.std(window_vals, ddof=1)) if w > 1 else 0.0
        else:
            feats[f"roll_mean_{w}"] = float("nan")
            feats[f"roll_std_{w}"] = float("nan")

    return feats

#04. Model factory (ML + Statistical)

In [0]:
def _get_ml_estimator(model_id: str, random_state: int = 42):

    from sklearn.ensemble import (
        RandomForestRegressor,
        ExtraTreesRegressor,
        GradientBoostingRegressor,
        AdaBoostRegressor,
    )
    from sklearn.linear_model import LinearRegression
    from sklearn.tree import DecisionTreeRegressor

    if model_id == "rf_cds_dt":
        return RandomForestRegressor(
            n_estimators=300, max_depth=12, min_samples_leaf=2, random_state=random_state, n_jobs=1, max_features="sqrt"
        )
    if model_id == "et_cds_dt":
        return ExtraTreesRegressor(
            n_estimators=400, random_state=random_state, n_jobs=1,
            min_samples_leaf=2, max_depth=12, max_features="sqrt"
        )
    if model_id == "gbr_cds_dt":
        return GradientBoostingRegressor(
            n_estimators=400, learning_rate=0.05, random_state=random_state,
            max_depth=3
        )
    if model_id == "ada_cds_dt":
        return AdaBoostRegressor(
            estimator=DecisionTreeRegressor(max_depth=4, random_state=random_state),
            n_estimators=500, learning_rate=0.05, random_state=random_state
        )
    if model_id == "ridge_cds_dt":
        
        return Ridge(alpha=10.0, random_state=random_state)
    
    if model_id == "lasso_cds_dt":
        return Lasso(alpha=1.0, random_state=random_state)

    if model_id == "lr_cds_dt":
        return LinearRegression()

    # optional libs
    if model_id == "lightgbm_cds_dt":
        import lightgbm as lgb
        return lgb.LGBMRegressor(
            n_estimators=600, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8,
            num_leaves=31, random_state=random_state, n_jobs=1
        )
    if model_id == "xgboost_cds_dt":
        import xgboost as xgb
        return xgb.XGBRegressor(
            n_estimators=600, learning_rate=0.03, max_depth=4, min_child_weight=3,
            subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=2.0, objective="reg:squarederror",
            n_jobs=1, random_state=random_state
        )
    if model_id == "catboost_cds_dt":
        from catboost import CatBoostRegressor
        return CatBoostRegressor(
            iterations=1200, learning_rate=0.03, depth=6,
            loss_function="MAE",
            verbose=False, random_seed=random_state,
            allow_writing_files=False,
            thread_count=1
        )

    raise ValueError(f"Unknown ML model_id: {model_id}")


def _fit_forecast_statistical(
    model_id: str,
    y_train: pd.Series,
    fh: int,
    seasonal_periods: Optional[int] = None,
) -> pd.Series:

    y_train = y_train.copy()
    y_train.index = pd.to_datetime(y_train.index)

    sp = int(seasonal_periods or SEASONAL_PERIODS_WINDOW or 1)

    # future index aligned with the configured frequency
    start = (y_train.index[-1] + OFFSET).normalize()
    idx = pd.date_range(start, periods=fh, freq=FREQ)

    if model_id == "snaive":
        hist = y_train.values.astype("float64", copy=False)
        preds = []
        for _ in range(fh):
            if sp > 1 and len(hist) >= sp:
                preds.append(hist[-sp])
            else:
                preds.append(hist[-1])
            hist = np.append(hist, preds[-1])
        return pd.Series(preds, index=idx, name="y_pred")

    if model_id in {"arima", "sarima"}:
        from statsmodels.tsa.arima.model import ARIMA

        order = (1, 1, 1)
        seasonal_order = (0, 0, 0, 0)
        if model_id == "sarima" and sp > 1:
            seasonal_order = (1, 1, 1, sp)

        model = ARIMA(y_train, order=order, seasonal_order=seasonal_order)
        fit = model.fit()
        preds = fit.forecast(fh)
        preds.index = idx
        return preds.rename("y_pred")

    if model_id == "prophet":
        from prophet import Prophet

        df_prophet = y_train.reset_index()
        df_prophet.columns = ["ds", "y"]
        df_prophet["ds"] = pd.to_datetime(df_prophet["ds"]).dt.tz_localize(None)

        trunc = str(SPARK_TRUNC).upper()
        model = Prophet(
            daily_seasonality=(trunc == "DAY"),
            weekly_seasonality=(trunc in {"DAY", "WEEK"}),
            yearly_seasonality=(trunc in {"DAY", "WEEK", "MONTH"}),
        )
        model.fit(df_prophet)

        future = model.make_future_dataframe(periods=fh, freq=FREQ)
        forecast = model.predict(future)
        preds = forecast["yhat"].iloc[-fh:].values
        return pd.Series(preds, index=idx, name="y_pred")

    if model_id == "tbats":
        from tbats import TBATS

        seasonal_list = [sp] if sp > 1 else []
        estimator = TBATS(seasonal_periods=seasonal_list)
        model = estimator.fit(y_train)
        preds = model.forecast(steps=fh)
        return pd.Series(preds, index=idx, name="y_pred")

    if model_id in {"ets", "exp_smooth", "theta"}:
        if model_id == "exp_smooth":
            from statsmodels.tsa.holtwinters import SimpleExpSmoothing
            fit = SimpleExpSmoothing(y_train, initialization_method="estimated").fit(optimized=True)
            preds = fit.forecast(fh)
            preds.index = idx
            return preds.rename("y_pred")

        if model_id == "ets":
            from statsmodels.tsa.holtwinters import ExponentialSmoothing
            use_seasonal = (sp > 1) and (len(y_train) >= 2 * sp)
            fit = ExponentialSmoothing(
                y_train,
                trend="add",
                damped_trend=True,
                seasonal="add" if use_seasonal else None,
                seasonal_periods=sp if use_seasonal else None,
                initialization_method="estimated",
            ).fit(optimized=True)
            preds = fit.forecast(fh)
            preds.index = idx
            return preds.rename("y_pred")

        if model_id == "theta":
            try:
                from statsmodels.tsa.forecasting.theta import ThetaModel
                tm = ThetaModel(y_train, period=sp if sp > 1 else 1)
                fit = tm.fit()
                preds = fit.forecast(fh)
                preds.index = idx
                return preds.rename("y_pred")
            except Exception:
                from statsmodels.tsa.holtwinters import ExponentialSmoothing
                fit = ExponentialSmoothing(
                    y_train,
                    trend="add",
                    damped_trend=True,
                    seasonal=None,
                    initialization_method="estimated",
                ).fit(optimized=True)
                preds = fit.forecast(fh)
                preds.index = idx
                return preds.rename("y_pred")

    raise ValueError(f"Unknown statistical model_id: {model_id}")


def _fit_forecast_ml(
    model_id: str,
    y_train: pd.Series,
    fh: int,
    max_lag: int | None = None,
    rolling_windows: Tuple[int, ...] | None = None,
    random_state: int = 42,
) -> pd.Series:

    if max_lag is None:
        max_lag = int(MAX_LAG_DEFAULT)
    if rolling_windows is None:
        rolling_windows = tuple(ROLLING_WINDOWS_DEFAULT)

    y_train = y_train.copy()
    y_train.index = pd.to_datetime(y_train.index)

    # --- build supervised matrix ---
    X, y_target, feature_cols = _build_supervised_matrix(
        y_train,
        max_lag=max_lag,
        rolling_windows=rolling_windows,
    )

    # remove time index feature for specific linear variants
    if model_id in ["lr_cds_dt", "ridge_cds_dt", "lasso_cds_dt"]:
        feature_cols = [c for c in feature_cols if c != "t"]
        X = X[feature_cols]

    # --- not enough rows after lagging -> fallback to naive ---
    if len(X) < 10:
        start = (y_train.index[-1] + OFFSET).normalize()
        idx = pd.date_range(start, periods=fh, freq=FREQ)
        return pd.Series([float(y_train.iloc[-1])] * fh, index=idx, name="y_pred")

    # --- fit estimator ---
    est = _get_ml_estimator(model_id, random_state=random_state)
    est.fit(X.values, y_target.values)

    # --- iterative forecasting ---
    history = list(y_train.values.astype("float64"))
    t_base = len(y_train) - 1

    start = (y_train.index[-1] + OFFSET).normalize()
    idx = pd.date_range(start, periods=fh, freq=FREQ)

    preds = []

    for step, dt in enumerate(idx, start=1):
        feats = _make_next_features(
            history=history,
            t_next=t_base + step,
            dt_next=dt,
            max_lag=max_lag,
            rolling_windows=rolling_windows,
        )

        # ensure feature order consistency
        xrow = pd.DataFrame([feats])[feature_cols].values
        yhat = float(est.predict(xrow)[0])

        # optional exponential decay adjustment for specific models
        if model_id in ["lr_cds_dt", "ridge_cds_dt", "lasso_cds_dt"]:
            yhat = yhat * (0.98 ** step)

        preds.append(yhat)
        history.append(yhat)

    return pd.Series(preds, index=idx, name="y_pred")


def _forecast_any_model(
    model_id: str,
    y_train: pd.Series,
    fh: int,
    seasonal_periods: Optional[int],
    clip_negative: bool = True,
) -> pd.Series:

    statistical_models = {"ets", "exp_smooth", "theta", "snaive", "arima", "sarima", "prophet", "tbats"}
    
    if model_id in statistical_models:
        preds = _fit_forecast_statistical(model_id, y_train, fh, seasonal_periods)
    else:
        preds = _fit_forecast_ml(model_id, y_train, fh)

    if clip_negative:
        preds = preds.clip(lower=0.0)
    return preds

# 05. Backtesting helpers

In [0]:
def _expanding_window_cutoffs(n: int, fh: int, folds: int, min_train: int) -> List[int]:

    if n < (min_train + fh):
        return []
    # Place folds towards the end, with non-overlapping test windows of length fh
    last_train_end = n - fh
    first_train_end = max(min_train, n - fh * folds)
    if first_train_end > last_train_end:
        first_train_end = last_train_end

    cutoffs = []
    # Evenly space cutoffs between first_train_end and last_train_end
    if folds == 1:
        cutoffs = [last_train_end]
    else:
        cutoffs = np.linspace(first_train_end, last_train_end, folds).round().astype(int).tolist()

    # Ensure strictly increasing and valid
    cutoffs = sorted(set([c for c in cutoffs if (c >= min_train and c + fh <= n)]))
    return cutoffs


def _backtest_mape(
    model_id: str,
    y: pd.Series,
    fh: int,
    folds: int,
    min_train: int,
    seasonal_periods: Optional[int],
) -> Optional[float]:

    n = len(y)
    cutoffs = _expanding_window_cutoffs(n, fh, folds, min_train)
    if not cutoffs:
        return None

    fold_mapes = []
    for c in cutoffs:
        y_train = y.iloc[:c]
        y_test = y.iloc[c:c + fh]
        preds = _forecast_any_model(model_id, y_train, fh=len(y_test), seasonal_periods=seasonal_periods)
        # align
        y_true = y_test.values
        y_pred = preds.values[:len(y_test)]
        fold_mapes.append(_safe_mape(y_true, y_pred))

    return float(np.mean(fold_mapes)) if fold_mapes else None

# 06. Main Spark function

In [0]:
def _floor_to_period_start(dt: pd.Series) -> pd.Series:

    s = pd.to_datetime(dt, errors="coerce")
    trunc = str(SPARK_TRUNC).upper()

    if trunc == "DAY":
        # normalize to midnight
        return s.dt.normalize()

    if trunc == "WEEK":
        # floor to Monday 00:00:00
        # dayofweek: Mon=0 ... Sun=6
        return (s.dt.normalize() - pd.to_timedelta(s.dt.dayofweek, unit="D"))

    if trunc == "MONTH":
        # first day of month 00:00:00
        # uses numpy datetime64[M] (safe, no Period)
        return pd.to_datetime(s.values.astype("datetime64[M]"))

    if trunc == "QUARTER":
        # first day of quarter 00:00:00 (safe, no Period)
        m = s.dt.month
        q_start_month = ((m.sub(1) // 3) * 3 + 1).astype("int64")  # 1,4,7,10
        return pd.to_datetime(
            dict(year=s.dt.year.astype("int64"), month=q_start_month, day=1)
        )

    if trunc == "YEAR":
        # Jan 1st 00:00:00
        return pd.to_datetime(dict(year=s.dt.year.astype("int64"), month=1, day=1))

    # fallback: just normalize
    return s.dt.normalize()

def _choose_recent_holdout(n: int, preferred: int, min_train_floor: int) -> int:

    if n <= 1:
        return 0
    fh = min(preferred, n - min_train_floor)
    if fh < 1:
        fh = 1
    return int(fh)

def _choose_split_windows(n: int, preferred_fh: int, min_train_default: int) -> tuple[int, int]:

    if n <= 1:
        return (max(0, n), 0)

    # try to keep some holdout, but never steal too much from a small split
    fh = min(preferred_fh, max(1, n // 4), n - 1)
    min_train = min_train_default

    # ensure feasible
    if min_train + fh > n:
        min_train = max(2, n - fh)  # keep at least 2 train if possible
        if min_train + fh > n:
            fh = max(1, n - min_train)

    # final safety
    min_train = min(min_train, n - 1)
    fh = min(fh, n - min_train)
    fh = max(1, fh)
    return int(min_train), int(fh)

def _backtest_mape_or_holdout(model_id, y, fh, folds, min_train, seasonal_periods, clip_negative=True):

    m = _backtest_mape(model_id, y, fh, folds, min_train, seasonal_periods)
    if m is not None:
        return float(m), "ok_cv"

    # fallback holdout
    n = len(y)
    if n <= 1:
        return None, "too_short"

    fh2 = min(fh, max(1, n - min_train))
    y_train = y.iloc[:-fh2]
    y_test  = y.iloc[-fh2:]
    preds = _forecast_any_model(model_id, y_train, fh=len(y_test), seasonal_periods=seasonal_periods, clip_negative=clip_negative)
    m2 = _safe_mape(y_test.values, preds.values[:len(y_test)])
    return float(m2), "ok_holdout"


In [0]:
def macro_forecast(
    sdf: DataFrame,
    dt_col: str = PERIOD_COLUMN,
    id_col: str = ID_COLUMN,
    y_col: str = Y_COLUMN,
    horizon: int = RANGE_PREDICTION,
    recent_validation: int = VALIDATION_RANGE,
    pandemic_start: str = PANDEMIC_START,
    pandemic_end: str = PANDEMIC_END,
    post_pandemic_start: str = POST_PANDEMIC_START,
    cv_folds: int = CV_FOLDS,
    min_train: int = MIN_TRAIN_PERIOD,
    fill_missing_value: float = FILL_MISSING_VALUES,
    clip_negative: bool = CLIP_NEGATIVE,
) -> DataFrame:

    candidate_models: List[str] = [
        "lightgbm_cds_dt",
        "xgboost_cds_dt",
        "catboost_cds_dt",
        "rf_cds_dt",
        "et_cds_dt",
        "gbr_cds_dt",
        "ada_cds_dt",
        "theta",
        "ets",
        "exp_smooth",
        "snaive",
        "ridge_cds_dt",
        "lasso_cds_dt",
        "lr_cds_dt",
        "arima",
        "sarima"#,
        #"prophet",
        #"tbats"
    ]

    base = (
        sdf.select(
            F.to_date(F.date_trunc(SPARK_TRUNC, F.to_timestamp(F.col(dt_col)))).alias("dt_period"),
            F.col(id_col).cast("string").alias("cd_key"),
            F.col(y_col).cast("double").alias("qt_units"),
        )
        .groupBy("cd_key", "dt_period")
        .agg(F.sum("qt_units").alias("qt_units"))
    )

    out_schema = T.StructType([
        T.StructField("cd_key", T.StringType(), False),
        T.StructField("record_type", T.StringType(), False),   # BACKTEST/RECENT_VALIDATION/SELECTION/FORECAST
        T.StructField("split", T.StringType(), True),          # Full/Pandemic/Post-Pandemic/RecentValidation/Forecast
        T.StructField("model_id", T.StringType(), True),
        T.StructField("effective_fh", T.IntegerType(), True),
        T.StructField("dt_period", T.DateType(), True),
        T.StructField("y_true", T.DoubleType(), True),
        T.StructField("y_pred", T.DoubleType(), True),
        T.StructField("mape", T.DoubleType(), True),
        T.StructField("winner_model_id", T.StringType(), True),
        T.StructField("winner_mape_recent", T.DoubleType(), True),
        T.StructField("status", T.StringType(), True),
    ])

    def _per_series(pdf: pd.DataFrame) -> pd.DataFrame:
        rows: List[Dict] = []
        sku = str(pdf["cd_key"].iloc[0])

        p = pdf.copy()
        p["dt_period"] = pd.to_datetime(p["dt_period"], errors="coerce")
        p = p.dropna(subset=["dt_period"])
        if p.empty:
            return pd.DataFrame([{
                "cd_key": sku, "record_type": "SELECTION", "split": "RecentValidation",
                "model_id": None, "effective_fh": None, "dt_period": None,
                "y_true": None, "y_pred": None, "mape": None,
                "winner_model_id": None, "winner_mape_recent": None,
                "status": "failed_empty"
            }])

        # Ensure start index
        p["dt_period"] = _floor_to_period_start(p["dt_period"])
        p = p.groupby("dt_period", as_index=False)["qt_units"].sum().sort_values("dt_period")

        full_idx = pd.date_range(p["dt_period"].min(), current_date, freq=FREQ)
        y = (
            p.set_index("dt_period")["qt_units"]
            .reindex(full_idx)
            .astype("float64")
            .fillna(fill_missing_value)
        )

        # Seasonal period
        seasonal_periods = SEASONAL_PERIODS_WINDOW if len(y) >= SEASONAL_PERIODS_WINDOW * 2 + 1 else None

    
        n_full = len(y)
        min_train_floor = max(min_train, 12)
        fh_recent = _choose_recent_holdout(n_full, preferred=recent_validation, min_train_floor=min_train_floor)

        ts_pandemic_start = pd.Timestamp(pandemic_start)
        ts_pandemic_end = pd.Timestamp(pandemic_end)
        ts_post_start = pd.Timestamp(post_pandemic_start)

        splits = {
            "Full": y,
            "Pandemic": y.loc[(y.index >= ts_pandemic_start) & (y.index <= ts_pandemic_end)],
            "Post-Pandemic": y.loc[y.index >= ts_post_start],
        }

        # 1) BACKTEST per split/model
        for split_name, y_split in splits.items():
            if len(y_split) <= 1:
                for mid in candidate_models:
                    rows.append({
                        "cd_key": sku, "record_type": "BACKTEST", "split": split_name,
                        "model_id": mid, "effective_fh": 0,
                        "dt_period": None, "y_true": None, "y_pred": None, "mape": None,
                        "winner_model_id": None, "winner_mape_recent": None,
                        "status": "too_short",
                    })
                continue

            min_train_s, fh_s = _choose_split_windows(len(y_split), preferred_fh=fh_recent, min_train_default=min_train)

            for mid in candidate_models:
                try:
                    mape_val, st = _backtest_mape_or_holdout(
                        model_id=mid,
                        y=y_split,
                        fh=fh_s,
                        folds=int(cv_folds),
                        min_train=min_train_s,
                        seasonal_periods=seasonal_periods,
                        clip_negative=clip_negative,
                    )
                    rows.append({
                        "cd_key": sku, "record_type": "BACKTEST", "split": split_name,
                        "model_id": mid, "effective_fh": int(fh_s),
                        "dt_period": None, "y_true": None, "y_pred": None, "mape": mape_val,
                        "winner_model_id": None, "winner_mape_recent": None,
                        "status": st if mape_val is not None else "failed_no_score",
                    })
                except ImportError:
                    rows.append({
                        "cd_key": sku, "record_type": "BACKTEST", "split": split_name,
                        "model_id": mid, "effective_fh": int(fh_recent),
                        "dt_period": None, "y_true": None, "y_pred": None, "mape": None,
                        "winner_model_id": None, "winner_mape_recent": None,
                        "status": "not_available",
                    })
                except Exception:
                    rows.append({
                        "cd_key": sku, "record_type": "BACKTEST", "split": split_name,
                        "model_id": mid, "effective_fh": int(fh_recent),
                        "dt_period": None, "y_true": None, "y_pred": None, "mape": None,
                        "winner_model_id": None, "winner_mape_recent": None,
                        "status": "failed",
                    })

        # 2) RECENT VALIDATION on last fh_recenT (Full only)
        winner_id: Optional[str] = None
        winner_mape: Optional[float] = None

        if len(y) <= 1:
            rows.append({
                "cd_key": sku, "record_type": "SELECTION", "split": "RecentValidation",
                "model_id": None, "effective_fh": 0,
                "dt_period": None, "y_true": None, "y_pred": None, "mape": None,
                "winner_model_id": None, "winner_mape_recent": None,
                "status": "too_short_full",
            })
            return pd.DataFrame(rows)

        # ensure fh_recent fits
        fh_recent = min(fh_recent, len(y) - 1)
        fh_recent = max(1, fh_recent)

        y_train = y.iloc[:-fh_recent]
        y_test = y.iloc[-fh_recent:]

        for mid in candidate_models:
            try:
                preds = _forecast_any_model(
                    model_id=mid,
                    y_train=y_train,
                    fh=int(fh_recent),
                    seasonal_periods=seasonal_periods,
                    clip_negative=clip_negative,
                )
                # Align by position
                mape = _safe_mape(y_test.values, preds.values[:len(y_test)])
                rows.append({
                    "cd_key": sku, "record_type": "RECENT_VALIDATION", "split": "RecentValidation",
                    "model_id": mid, "effective_fh": int(fh_recent),
                    "dt_period": None,
                    "y_true": float(np.mean(y_test.values)),
                    "y_pred": float(np.mean(preds.values)),
                    "mape": float(mape),
                    "winner_model_id": None, "winner_mape_recent": None,
                    "status": "ok",
                })

                if (winner_mape is None) or (mape < winner_mape):
                    winner_mape = float(mape)
                    winner_id = mid

            except ImportError:
                rows.append({
                    "cd_key": sku, "record_type": "RECENT_VALIDATION", "split": "RecentValidation",
                    "model_id": mid, "effective_fh": int(fh_recent),
                    "dt_period": None, "y_true": None, "y_pred": None, "mape": None,
                    "winner_model_id": None, "winner_mape_recent": None,
                    "status": "not_available",
                })
            except Exception:
                rows.append({
                    "cd_key": sku, "record_type": "RECENT_VALIDATION", "split": "RecentValidation",
                    "model_id": mid, "effective_fh": int(fh_recent),
                    "dt_period": None, "y_true": None, "y_pred": None, "mape": None,
                    "winner_model_id": None, "winner_mape_recent": None,
                    "status": "failed",
                })

        # 3) SELECTION row
        rows.append({
            "cd_key": sku, "record_type": "SELECTION", "split": "RecentValidation",
            "model_id": None, "effective_fh": int(fh_recent),
            "dt_period": None, "y_true": None, "y_pred": None, "mape": winner_mape,
            "winner_model_id": winner_id, "winner_mape_recent": winner_mape,
            "status": "ok" if winner_id is not None else "no_winner",
        })

        if winner_id is None:
            return pd.DataFrame(rows)

        # 4) FINAL FORECAST ahead using winner on full series
        try:
            preds_120 = _forecast_any_model(
                model_id=winner_id,
                y_train=y,
                fh=int(horizon),
                seasonal_periods=seasonal_periods,
                clip_negative=clip_negative,
            )
            for dt, val in zip(preds_120.index, preds_120.values):
                rows.append({
                    "cd_key": sku, "record_type": "FORECAST", "split": "Forecast",
                    "model_id": winner_id, "effective_fh": int(horizon),
                    "dt_period": pd.Timestamp(dt).date(),
                    "y_true": None, "y_pred": float(val),
                    "mape": None,
                    "winner_model_id": winner_id, "winner_mape_recent": winner_mape,
                    "status": "ok",
                })
        except ImportError:
            rows.append({
                "cd_key": sku, "record_type": "FORECAST", "split": "Forecast",
                "model_id": winner_id, "effective_fh": int(horizon),
                "dt_period": None, "y_true": None, "y_pred": None, "mape": None,
                "winner_model_id": winner_id, "winner_mape_recent": winner_mape,
                "status": "not_available",
            })
        except Exception:
            rows.append({
                "cd_key": sku, "record_type": "FORECAST", "split": "Forecast",
                "model_id": winner_id, "effective_fh": int(horizon),
                "dt_period": None, "y_true": None, "y_pred": None, "mape": None,
                "winner_model_id": winner_id, "winner_mape_recent": winner_mape,
                "status": "failed_forecast",
            })

        return pd.DataFrame(rows)

    return base.groupBy("cd_key").applyInPandas(_per_series, schema=out_schema)


# Optional helpers
def get_forecast_df(result_df: DataFrame) -> DataFrame:
    return result_df.filter(F.col("record_type") == F.lit("FORECAST"))


def get_metrics_df(result_df: DataFrame) -> DataFrame:
    return result_df.filter(F.col("record_type").isin(["BACKTEST", "RECENT_VALIDATION", "SELECTION"]))


In [0]:
done_keys: set[str] = set()
failed_keys: set[str] = set()

def _next_key_to_run(df_input):
    candidates = (
        df_input.select("cd_key").distinct()
        .filter(~F.col("cd_key").isin(list(done_keys | failed_keys)))
        .limit(1)
        .collect()
    )
    return candidates[0]["cd_key"] if candidates else None


while True:
    df_input = create_input()

    if df_input is None or df_input.isEmpty():
        print("No input rows. Stopping.")
        break

    cd_key = _next_key_to_run(df_input)

    if cd_key is None:
        print("No remaining cd_key to process (all done or failed). Stopping.")
        break

    try:
        result = macro_forecast(df_input)

        forecast_df = get_forecast_df(result)
        metrics_df  = get_metrics_df(result)

        forecast_df.write.format("delta").mode("append").saveAsTable(MACRO_FORECASTING_TABLE)

        done_keys.add(cd_key)

    except Exception as e:
        failed_keys.add(cd_key)
        print(f"FAILED: {cd_key} -> {e}")
        continue